# Test Symmetric Dirichlet Parametrization using MeshFEM's New `MeshEnergy` Class

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys, os
sys.path.append('../')
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, benchmark
import energy

import param_utils
import helper_funcs

In [ ]:
# m = mesh.Mesh('../../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh')
m = mesh.Mesh('../../3rdparty/MeshFEM/misc/examples/meshes/uv-100.obj')

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
bdry_uv = helper_funcs.getBDdataOnUnitCircle(m)

In [ ]:
# Tutte Initialization
# uv.setVars(parametrization.lscm(m).ravel())
uv_init = parametrization.harmonic(m, bdry_uv, False)
flip_list = parametrization.getFlips(m, uv_init)
if len(flip_list) > 0:  uv_init = parametrization.harmonic(m, bdry_uv, True)

In [ ]:
uv.setVars(uv_init.ravel())

In [ ]:
e = energy.SymmetricDirichlet(2)

In [ ]:
# Construct `SymmetricDirichlet` parametrization energy and problem
# param = mesh_energy.Parametrization(m, uv, e)
param = mesh_energy.SymmDriParametrization(m, uv, e)
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [param])

In [ ]:
em = MeshFEM.EmbeddedMesh(m, uv)
v = viewer.Viewer(em, wireframe=True)
v.show()

In [ ]:
# Work around energy nullspace by adding a small shift
prob.hessianShift = 1e-8
opt = prob.optimizer()

In [ ]:
opt.options.niter = 200

In [ ]:
benchmark.reset()
opt.optimize()
benchmark.report()

In [ ]:
v.update()

In [ ]:
param_viewer = param_utils.ParametrizationViewer(m, uv.getVars().reshape(-1,2))
param_viewer.show()